# IITGN Cohort 2 — Python Coding Assignment

**Course:** PG Diploma in AI/ML and Agentic AI  
**Institute:** IIT Gandhinagar  
**Student Name:** Vysag Karunakaran  
**Student ID:** 26271031

---

This notebook contains solutions for Questions 1–10 of the Python Problem Solving for AI and ML assignment.

# Question 1 — Clean Training Records Before Dataset Preparation

## Approach

For each record:

1. Check that the record is a dictionary.
2. Check that `learner_id` and `score` exist.
3. Validate and trim the learner ID.
4. Validate the score type.
5. Convert valid scores to `float`.
6. Reject non-finite values and values outside 0–100.
7. Mark the learner as eligible when the score is at least 60.
8. Store invalid record positions in `rejected_indices`.

The input records are not modified.

In [2]:
import math


def clean_training_records(records):
    cleaned = []
    rejected_indices = []

    for index, record in enumerate(records):

        # Record must be a dictionary
        if not isinstance(record, dict):
            rejected_indices.append(index)
            continue

        # Required keys must exist
        if "learner_id" not in record or "score" not in record:
            rejected_indices.append(index)
            continue

        learner_id = record["learner_id"]
        score = record["score"]

        # Validate learner ID
        if not isinstance(learner_id, str):
            rejected_indices.append(index)
            continue

        learner_id = learner_id.strip()

        if not learner_id:
            rejected_indices.append(index)
            continue

        # Boolean values must not be treated as numbers
        if isinstance(score, bool):
            rejected_indices.append(index)
            continue

        # Only int, float, or convertible string scores are allowed
        if not isinstance(score, (int, float, str)):
            rejected_indices.append(index)
            continue

        try:
            score = float(score)
        except (ValueError, TypeError):
            rejected_indices.append(index)
            continue

        # Reject NaN and infinity
        if not math.isfinite(score):
            rejected_indices.append(index)
            continue

        # Score must be between 0 and 100
        if not 0 <= score <= 100:
            rejected_indices.append(index)
            continue

        cleaned.append({
            "learner_id": learner_id,
            "score": score,
            "eligible": score >= 60
        })

    return {
        "cleaned": cleaned,
        "rejected_indices": rejected_indices
    }

In [3]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": " ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

clean_training_records(records)

{'cleaned': [{'learner_id': 'L01', 'score': 82.5, 'eligible': True},
  {'learner_id': 'L02', 'score': 60.0, 'eligible': True},
  {'learner_id': 'L05', 'score': 48.0, 'eligible': False}],
 'rejected_indices': [2, 3, 4, 6]}

# Question 2 — Prepare Mixed Text Payloads for an NLP Pipeline

## Approach

For each payload:

1. Accept normal strings directly.
2. Decode `bytes` and `bytearray` using UTF-8.
3. Reject unsupported types and binary data that cannot be decoded.
4. Convert accepted text to lowercase.
5. Split the text on whitespace.
6. Remove only `. , ! ? ; :` from the beginning and end of each token.
7. Keep only tokens whose length is at least `min_length`.
8. Preserve token order and duplicates.
9. Build a sorted unique vocabulary across all accepted payloads.

The function also validates that `min_length` is a positive integer and does not modify the input objects.

In [ ]:
def prepare_messages(payloads, min_length=3):

    # Validate min_length
    if (
        isinstance(min_length, bool)
        or not isinstance(min_length, int)
        or min_length <= 0
    ):
        raise ValueError("min_length must be a positive integer")

    token_lists = []
    rejected_indices = []

    for index, payload in enumerate(payloads):

        # Accept normal strings directly
        if isinstance(payload, str):
            text = payload

        # Decode bytes and bytearray
        elif isinstance(payload, (bytes, bytearray)):
            try:
                text = payload.decode("utf-8")
            except UnicodeDecodeError:
                rejected_indices.append(index)
                continue

        # Reject unsupported types
        else:
            rejected_indices.append(index)
            continue

        # Convert to lowercase and split into tokens
        raw_tokens = text.lower().split()

        # List comprehension:
        # remove only specified punctuation from token ends
        cleaned_tokens = [
            token.strip(".,!?;:")
            for token in raw_tokens
        ]

        # Keep tokens meeting minimum length
        tokens = [
            token
            for token in cleaned_tokens
            if len(token) >= min_length
        ]

        token_lists.append(tokens)

    # Set comprehension:
    # create unique vocabulary
    vocabulary = sorted({
        token
        for tokens in token_lists
        for token in tokens
    })

    return {
        "token_lists": token_lists,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices
    }

In [ ]:
payloads = [
    " AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]

prepare_messages(payloads, min_length=3)

# Question 3 — Summarise Classification Prediction Results

## Approach

For each prediction record:

1. Check that the record is a dictionary.
2. Check that `id`, `actual`, and `predicted` are present.
3. Ensure all three required values are non-empty strings.
4. Trim surrounding whitespace.
5. Convert `actual` and `predicted` labels to lowercase.
6. Count valid records and correct predictions.
7. Build a summary for each actual class.
8. Store IDs of incorrect predictions in input order.
9. Store rejected record positions using zero-based indexing.
10. Calculate accuracy as `correct_count / valid_count`, rounded to four decimal places.

If there are no valid records, accuracy is `None`.

In [ ]:
def summarise_predictions(results):

    valid_count = 0
    correct_count = 0

    by_actual = {}
    misclassified_ids = []
    rejected_indices = []

    required_keys = ("id", "actual", "predicted")

    for index, record in enumerate(results):

        # Record must be a dictionary
        if not isinstance(record, dict):
            rejected_indices.append(index)
            continue

        # All required keys must exist
        if any(key not in record for key in required_keys):
            rejected_indices.append(index)
            continue

        # All required values must be strings
        if any(
            not isinstance(record[key], str)
            for key in required_keys
        ):
            rejected_indices.append(index)
            continue

        record_id = record["id"].strip()
        actual = record["actual"].strip()
        predicted = record["predicted"].strip()

        # Reject blank strings
        if not record_id or not actual or not predicted:
            rejected_indices.append(index)
            continue

        # Normalize labels
        actual = actual.lower()
        predicted = predicted.lower()

        valid_count += 1

        # Create category entry if needed
        if actual not in by_actual:
            by_actual[actual] = {
                "total": 0,
                "correct": 0
            }

        by_actual[actual]["total"] += 1

        # Check prediction
        if actual == predicted:
            correct_count += 1
            by_actual[actual]["correct"] += 1
        else:
            misclassified_ids.append(record_id)

    # Calculate accuracy
    if valid_count == 0:
        accuracy = None
    else:
        accuracy = round(
            correct_count / valid_count,
            4
        )

    return {
        "valid_count": valid_count,
        "correct_count": correct_count,
        "accuracy": accuracy,
        "by_actual": by_actual,
        "misclassified_ids": misclassified_ids,
        "rejected_indices": rejected_indices
    }

In [ ]:
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

summarise_predictions(results)

# Question 4 — Detect Dataset Duplicates and Overlap

## Approach

For both the training and validation ID lists:

1. Accept only string identifiers.
2. Remove surrounding whitespace.
3. Reject blank strings and non-string values.
4. Record invalid positions using zero-based indexing.
5. Detect identifiers that occur more than once.
6. Convert valid identifiers to sets for dataset comparison.
7. Find:
   - identifiers shared by both datasets,
   - identifiers only in training,
   - identifiers only in validation.
8. Sort all returned identifier lists in ascending order.
9. Create `frozenset` snapshots of the valid unique IDs.

The original input lists are not modified.

In [ ]:
def audit_dataset_ids(train_ids, validation_ids):

    def process_ids(values):

        valid_ids = []
        invalid_indices = []

        # Validate and clean
        for index, value in enumerate(values):

            if not isinstance(value, str):
                invalid_indices.append(index)
                continue

            cleaned = value.strip()

            if not cleaned:
                invalid_indices.append(index)
                continue

            valid_ids.append(cleaned)

        # Count occurrences
        counts = {}

        for value in valid_ids:
            counts[value] = counts.get(value, 0) + 1

        # Find duplicates
        duplicates = sorted([
            value
            for value, count in counts.items()
            if count > 1
        ])

        return valid_ids, duplicates, invalid_indices

    # Process training IDs
    train_valid, train_duplicates, invalid_train = process_ids(train_ids)

    # Process validation IDs
    validation_valid, validation_duplicates, invalid_validation = process_ids(
        validation_ids
    )

    # Convert to sets for comparisons
    train_set = set(train_valid)
    validation_set = set(validation_valid)

    return {
        "train_duplicates": train_duplicates,
        "validation_duplicates": validation_duplicates,

        "overlap": sorted(train_set & validation_set),

        "train_only": sorted(
            train_set - validation_set
        ),

        "validation_only": sorted(
            validation_set - train_set
        ),

        "invalid_train_indices": invalid_train,
        "invalid_validation_indices": invalid_validation,

        "snapshots": {
            "train": frozenset(train_set),
            "validation": frozenset(validation_set)
        }
    }

In [ ]:
train_ids = ["R3", "R1", " R2 ", "R1", None]

validation_ids = [
    "R2",
    "R4",
    "R4",
    " "
]

audit_dataset_ids(
    train_ids,
    validation_ids
)

# Question 5 — Update an Annotation Queue

## Approach

The function updates an annotation queue without modifying the original input lists.

Steps:

1. Combine `queue` and `incoming`.
2. Keep only the first occurrence of each document ID.
3. Remove all IDs that appear in `completed`.
4. If `priority` is present in the remaining queue, move it to the front.
5. Use slicing to create the next batch.
6. Use indexing to get the final item.
7. Use `range()` to create numbered `(position, document_id)` tuples.

The function also validates that `batch_size` is a non-negative integer excluding Boolean values.

In [ ]:
def update_annotation_queue(
    queue,
    incoming,
    completed,
    priority=None,
    batch_size=3
):

    # Validate batch_size
    if (
        isinstance(batch_size, bool)
        or not isinstance(batch_size, int)
        or batch_size < 0
    ):
        raise ValueError("batch_size must be a non-negative integer")

    updated_queue = []

    # Combine queue and incoming
    # Keep only first occurrence of each ID
    for document_id in queue + incoming:
        if document_id not in updated_queue:
            updated_queue.append(document_id)

    # Remove completed IDs
    for document_id in completed:
        if document_id in updated_queue:
            updated_queue.remove(document_id)

    # Move priority ID to the front if present
    if priority in updated_queue:
        updated_queue.remove(priority)
        updated_queue.insert(0, priority)

    # Slicing for next batch
    next_batch = updated_queue[:batch_size]

    # Indexing for final item
    last_item = updated_queue[-1] if updated_queue else None

    # Numbered positions starting at 1
    positions = [
        (index + 1, updated_queue[index])
        for index in range(len(updated_queue))
    ]

    return {
        "queue": updated_queue,
        "next_batch": next_batch,
        "last_item": last_item,
        "positions": positions
    }

In [ ]:
queue = ["D3", "D1", "D3", "D2"]

incoming = ["D4", "D2", "D5"]

completed = ["D1", "D9"]

update_annotation_queue(
    queue,
    incoming,
    completed,
    priority="D5",
    batch_size=3
)

# Question 6 — Build Independent Experiment Configurations

## Approach

The function creates a new experiment configuration from a baseline configuration.

Steps:

1. Validate that `run_name` is a non-empty string after trimming.
2. Validate that `parameter_updates` is either a dictionary or `None`.
3. Create an independent deep copy of the base configuration.
4. Apply parameter updates only to keys already present in `base["parameters"]`.
5. Raise `KeyError` if an unknown parameter is supplied.
6. Merge extra keyword arguments into the copied metadata dictionary.
7. Add the trimmed `run_name` at the top level.
8. Return the new configuration without modifying the original base dictionary.

In [ ]:
from copy import deepcopy
from typing import Any


def build_experiment_config(
    base: dict,
    *,
    run_name: str,
    parameter_updates: dict | None = None,
    **metadata: Any
) -> dict:

    # Validate run_name
    if not isinstance(run_name, str) or not run_name.strip():
        raise ValueError("run_name must be a non-empty string")

    # Validate parameter_updates
    if parameter_updates is not None and not isinstance(
        parameter_updates,
        dict
    ):
        raise TypeError("parameter_updates must be a dictionary or None")

    # Create independent deep copy
    config = deepcopy(base)

    # Apply parameter updates if provided
    if parameter_updates:

        # Validate all parameter names first
        for key in parameter_updates:
            if key not in base["parameters"]:
                raise KeyError(key)

        config["parameters"].update(parameter_updates)

    # Merge extra metadata
    config["metadata"].update(metadata)

    # Add trimmed run name
    config["run_name"] = run_name.strip()

    return config

In [ ]:
base = {
    "model": "ticket_classifier",
    "parameters": {
        "threshold": 0.5,
        "max_tokens": 128
    },
    "metadata": {
        "owner": "platform",
        "stage": "baseline"
    }
}

result = build_experiment_config(
    base,
    run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team",
    seed=42
)

result

# Question 7 — Aggregate Evaluation Scores Across Batches

## Approach

The function accepts any number of score batches using `*batches`.

Steps:

1. Validate `top_n`.
2. Check that every batch is either a list or tuple.
3. Inspect every score along with its batch index and item index.
4. Accept only finite integers or floats from 0 to 100.
5. Explicitly reject Boolean values.
6. Store invalid score positions as `(batch_index, item_index)`.
7. Calculate the mean of valid scores using a generator expression.
8. Sort valid scores by:
   - score descending,
   - batch index ascending,
   - item index ascending.
9. Return only the first `top_n` ranked entries.

The original batches are not modified.

In [ ]:
import math


def aggregate_scores(*batches, top_n=3):

    # Validate top_n
    if (
        isinstance(top_n, bool)
        or not isinstance(top_n, int)
        or top_n < 0
    ):
        raise ValueError("top_n must be a non-negative integer")

    valid_scores = []
    rejected_positions = []

    # Process every batch
    for batch_index, batch in enumerate(batches):

        # Every batch must be a list or tuple
        if not isinstance(batch, (list, tuple)):
            raise TypeError("Each batch must be a list or tuple")

        # Process every score in the batch
        for item_index, score in enumerate(batch):

            is_valid = (
                not isinstance(score, bool)
                and isinstance(score, (int, float))
                and math.isfinite(score)
                and 0 <= score <= 100
            )

            if is_valid:
                valid_scores.append(
                    (
                        batch_index,
                        item_index,
                        float(score)
                    )
                )
            else:
                rejected_positions.append(
                    (
                        batch_index,
                        item_index
                    )
                )

    valid_count = len(valid_scores)

    # Generator expression used here
    if valid_count > 0:
        mean_score = round(
            sum(score for _, _, score in valid_scores)
            / valid_count,
            2
        )
    else:
        mean_score = None

    # Sort by:
    # 1. score descending
    # 2. batch index ascending
    # 3. item index ascending
    ranked_scores = sorted(
        valid_scores,
        key=lambda item: (
            -item[2],
            item[0],
            item[1]
        )
    )

    return {
        "valid_count": valid_count,
        "mean": mean_score,
        "top_scores": ranked_scores[:top_n],
        "rejected_positions": rejected_positions
    }

# Question 8 — Organise Agent Tool Call Records

## Approach

The function processes agent tool-call records using a last-record-wins rule.

Steps:

1. Keep only the final record for each `call_id`.
2. Group the final records by tool.
3. For each tool, calculate:
   - total calls
   - successful calls
   - failed calls
   - total duration in milliseconds
4. Collect failed call IDs and sort them alphabetically.
5. Rank tools by:
   - failure count descending
   - total calls descending
   - tool name ascending
6. Use a dictionary comprehension for part of the report construction.

The original input records are not modified.

In [ ]:
def summarise_tool_calls(records):

    # Keep only the final record for each call_id
    latest_calls = {}

    for record in records:
        latest_calls[record["call_id"]] = record

    # Get unique tool names
    tool_names = {
        record["tool"]
        for record in latest_calls.values()
    }

    # Dictionary comprehension required by the question
    by_tool = {
        tool: {
            "calls": 0,
            "successes": 0,
            "failures": 0,
            "total_duration_ms": 0
        }
        for tool in tool_names
    }

    failed_call_ids = []

    # Summarise only final records
    for call_id, record in latest_calls.items():

        tool = record["tool"]
        status = record["status"]
        duration = record["duration_ms"]

        by_tool[tool]["calls"] += 1
        by_tool[tool]["total_duration_ms"] += duration

        if status == "ok":
            by_tool[tool]["successes"] += 1
        else:
            by_tool[tool]["failures"] += 1
            failed_call_ids.append(call_id)

    # Sort failed IDs alphabetically
    failed_call_ids.sort()

    # Rank tools:
    # 1. failures descending
    # 2. calls descending
    # 3. tool name ascending
    tool_ranking = sorted(
        by_tool,
        key=lambda tool: (
            -by_tool[tool]["failures"],
            -by_tool[tool]["calls"],
            tool
        )
    )

    return {
        "unique_calls": len(latest_calls),
        "by_tool": by_tool,
        "failed_call_ids": failed_call_ids,
        "tool_ranking": tool_ranking
    }

# Question 9 — Validate Text Batches with a Reusable Decorator

## Approach

This question uses a decorator to apply the same validation rules to text-processing functions.

The decorator will:

1. Accept `texts` either as the first positional argument or as a keyword argument.
2. Require `texts` to be a list.
3. Require every item in the list to be a string.
4. Reject strings that are empty after whitespace removal.
5. Validate the complete batch before calling the wrapped function.
6. Preserve the wrapped function's name and docstring using `functools.wraps`.
7. Forward all other positional and keyword arguments unchanged.

The decorated function `text_lengths` returns character counts:

- after trimming surrounding whitespace when `strip=True`
- from the original strings when `strip=False`

In [ ]:
from functools import wraps


def validate_text_batch(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        # Get texts either positionally or by keyword
        if args:
            texts = args[0]
        elif "texts" in kwargs:
            texts = kwargs["texts"]
        else:
            # Let the wrapped function handle a missing argument normally
            return func(*args, **kwargs)

        # texts must be a list
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        # Check all item types first
        if any(not isinstance(text, str) for text in texts):
            raise TypeError("Every item must be a string")

        # Then check for blank strings
        if any(not text.strip() for text in texts):
            raise ValueError("Blank strings are not allowed")

        # Input is valid, so call the original function
        return func(*args, **kwargs)

    return wrapper


@validate_text_batch
def text_lengths(
    texts: list[str],
    *,
    strip: bool = True
) -> list[int]:
    """Return the length of each text item."""

    if strip:
        return [len(text.strip()) for text in texts]

    return [len(text) for text in texts]

In [ ]:
texts = [" agent ", "ML", "data"]

text_lengths(texts)

# Question 10 — Track Successful and Failed Function Calls

## Approach

This question uses a decorator factory called `track_calls(history)`.

The decorator will:

1. Receive a history list.
2. Wrap any function that uses the decorator.
3. Assign a call number starting from 1.
4. Execute the wrapped function.
5. If the function succeeds:
   - record status as `"success"`
   - record `error_type` as `None`
   - return the original result unchanged
6. If the function raises an exception:
   - record status as `"error"`
   - record the exception class name
   - re-raise the original exception
7. Preserve the original function's name and docstring using `functools.wraps`.

The shared history list is intentionally modified by the decorator.

In [ ]:
from functools import wraps
import math


def track_calls(history):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            call_number = len(history) + 1

            try:
                result = func(*args, **kwargs)

            except Exception as error:

                history.append({
                    "call_number": call_number,
                    "function": func.__name__,
                    "status": "error",
                    "error_type": type(error).__name__
                })

                raise

            history.append({
                "call_number": call_number,
                "function": func.__name__,
                "status": "success",
                "error_type": None
            })

            return result

        return wrapper

    return decorator